# 01 · Preprocessing
**Input:** Raw MTX files from GEO accession GSE166635
**Output:** `data/processed/adata_processed.h5ad`

Covers: data loading, QC metrics, cell filtering, normalisation, log-transform,
and highly variable gene selection.


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:

# ── Path configuration ─────────────────────────────────────────────────────────
# Edit DATA_DIR to point to where you extracted GSE166635_RAW/
# All other paths in this notebook are relative to the repo root.
from pathlib import Path

DATA_DIR    = Path("../data/raw")        # HCC1/ and HCC2/ live here
RESULTS_DIR = Path("../results")
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR  = RESULTS_DIR / "tables"

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## Load data

In [ ]:
# Load each sample from the MTX triplet
adata1 = sc.read_10x_mtx(DATA_DIR / "HCC1", var_names="gene_symbols")
adata2 = sc.read_10x_mtx(DATA_DIR / "HCC2", var_names="gene_symbols")

adata1.obs["sample"] = "HCC1"
adata2.obs["sample"] = "HCC2"

adata1.var_names_make_unique()
adata2.var_names_make_unique()

adata = adata1.concatenate(adata2, batch_key="sample")
adata.obs["sample"] = adata.obs["sample"].map({"0": "normal (HCC1)", "1": "tumor (HCC2)"})
print(adata)
print(adata.obs["sample"].value_counts())

## Quality control

In [ ]:
def qc_metrics(adata):
    adata.var["mt"]   = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    adata.var["hb"]   = adata.var_names.str.contains("^HB[^(P)]")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt","ribo","hb"],
                                inplace=True, log1p=True)
    return adata

adata = qc_metrics(adata)

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts","total_counts","pct_counts_mt"],
             groupby="sample", multi_panel=True,
             save="_qc_violin.png")
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="sample")

## Cell filtering

In [ ]:
def filter_cells(adata, min_genes=200, max_genes=2500, max_mt_pct=5):
    print(f"Before filtering: {adata.n_obs} cells")
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_cells(adata, max_genes=max_genes)
    adata = adata[adata.obs["pct_counts_mt"] < max_mt_pct]
    print(f"After filtering : {adata.n_obs} cells")
    return adata

adata = filter_cells(adata)

## Normalisation & log-transform

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"Max expression after log1p: {adata.X.max():.2f}")

## Feature selection — highly variable genes

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")
sc.pl.highly_variable_genes(adata)
print(f"HVGs selected: {adata.var.highly_variable.sum()}")

## Save processed object

In [ ]:
out_path = Path("../data/processed/adata_processed.h5ad")
out_path.parent.mkdir(parents=True, exist_ok=True)
adata.write(out_path)
print(f"Saved: {out_path}  ({adata.n_obs} cells x {adata.n_vars} genes)")